In [25]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

base_dir = Path().resolve().parent
sys.path.append(str(base_dir))

from src.data_preprocessing import data_split
from utils.evaluate_model import evaluate_model
from src.shap_plots import compute_shap
from utils.save_results import save_evaluation, save_shap, save_model

RF_MODEL_DIR = str(base_dir / "models_sectors/rf")
RIDGE_MODEL_DIR = str(base_dir / "models_sectors/ridge")
RF_MODEL_PARAMS = str(base_dir / "params/rf")
RIDGE_MODEL_PARAMS = str(base_dir / "params/ridge")


In [26]:
FEATURE_COLS = [
    "CPI_Change_lag1", "Rate_Change", "GDP_Growth_lag2",
    "Unemp_Change_lag1", "USD_Change", "VIX_Change",
    "Credit_Spread_lag2",
]

SECTORS = {
    "tech":       "Tech_Return",
    "healthcare": "Healthcare_Return",
    "finance":    "Finance_Return",
    "industrial": "Industrial_Return",
    "energy":     "Energy_Return",
}

# load data
df = pd.read_csv("../data/processed/processed_data.csv", parse_dates=['Date'])
print(f"Data shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.isnull().sum()
df.dropna(inplace=True)  # drop rows with missing values (if any)
print(f"Data shape after dropping NA: {df.shape}")


Data shape: (415, 18)
Columns: ['Date', 'SP500_Return', 'Tech_Return', 'Healthcare_Return', 'Finance_Return', 'Industrial_Return', 'Energy_Return', 'CPI_Change', 'Rate_Change', 'GDP_Growth', 'Unemp_Change', 'USD_Change', 'VIX_Change', 'Credit_Spread', 'CPI_Change_lag1', 'GDP_Growth_lag2', 'Unemp_Change_lag1', 'Credit_Spread_lag2']
Data shape after dropping NA: (321, 18)


In [27]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [28]:
all_rf_params = {}
all_rf_results = {}

for sector_name, return_col in SECTORS.items():

    print(f"\n{'='*55}")
    print(f"  {sector_name.upper()} → {return_col}")
    print(f"{'='*55}")

    # Split data
    splits = data_split(df, FEATURE_COLS, return_col)
    X_train = splits["X_train"]
    y_train = splits["y_train"]
    X_test  = splits["X_test"]
    y_test  = splits["y_test"]
    print(f"  Train: {len(X_train)} | Test: {len(X_test)}")

    # Tune with Optuna (same setup as previous RF notebook)
    def objective(trial):
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 30, 100, step=5),
            "max_depth":         trial.suggest_int("max_depth", 2, 6),
            "min_samples_split": trial.suggest_int("min_samples_split", 5, 50),
            "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 3, 30),
            "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]),
            "max_samples":       trial.suggest_float("max_samples", 0.3, 0.9),
            "n_jobs": -1,
            "random_state": 42,
        }

        model = RandomForestRegressor(**params)
        model.fit(X_train, y_train)

        test_pred = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, test_pred))
        return rmse

    study = optuna.create_study(
        direction="minimize",
        study_name=f"rf_{sector_name}",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=200, show_progress_bar=True)

    best = study.best_params
    best["n_jobs"] = -1
    best["random_state"] = 42
    all_rf_params[sector_name] = best
    print(f"  Best RMSE: {study.best_value:.4f}")
    print(f"  Params: {best}")

    # Train final model with best params
    model = RandomForestRegressor(**best)
    model.fit(X_train, y_train)

    # Evaluate on train and test
    ev_result = evaluate_model(
        model, X_train, y_train, X_test, y_test,
        f"RF - {sector_name}"
    )

    # Compute SHAP
    shap_test_df, mean_abs_shap = compute_shap(
        model, X_train, X_test, FEATURE_COLS, model_type="tree"
    )

    # save
    save_evaluation(sector_name, ev_result, FEATURE_COLS, y_test, RF_MODEL_DIR)
    save_shap(sector_name, shap_test_df, mean_abs_shap, FEATURE_COLS, RF_MODEL_DIR)
    save_model(sector_name, model, RF_MODEL_DIR)

    all_rf_results[sector_name] = {
        "best_rmse": study.best_value,
        "test_r2": ev_result[1]["r2"],
        "test_rmse": ev_result[1]["rmse"],
        "test_dir": ev_result[1]["dir"],
    }

print("\n ALL SECTORS TRAINED AND SAVED")


  TECH → Tech_Return

  Features (7): CPI_Change_lag1, Rate_Change, GDP_Growth_lag2, Unemp_Change_lag1, USD_Change, VIX_Change, Credit_Spread_lag2
  Train: 252 | Test: 69


Best trial: 147. Best value: 5.39444: 100%|██████████| 200/200 [00:13<00:00, 15.03it/s]


  Best RMSE: 5.3944
  Params: {'n_estimators': 90, 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 0.7, 'max_samples': 0.6148378737064784, 'n_jobs': -1, 'random_state': 42}
  RF - tech
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.3525     0.2629
  RMSE (%)                       5.407      5.394
  MAE (%)                        3.811      4.333
  Directional Acc (%)            72.62      71.01
  Base value (mean prediction) : 0.4093

  Saving evaluation results for /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results...
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results/tech_results.pkl
R² train=0.3525  test=0.2629
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/shap/tech_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/Desktop/macroeconomi

Best trial: 183. Best value: 4.00932: 100%|██████████| 200/200 [00:12<00:00, 15.48it/s]


  Best RMSE: 4.0093
  Params: {'n_estimators': 90, 'max_depth': 6, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 0.5, 'max_samples': 0.377554823132828, 'n_jobs': -1, 'random_state': 42}
  RF - healthcare
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.4454     0.1284
  RMSE (%)                       2.992      4.009
  MAE (%)                        2.330      3.163
  Directional Acc (%)            75.79      63.77
  Base value (mean prediction) : 0.5493

  Saving evaluation results for /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results...
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results/healthcare_results.pkl
R² train=0.4454  test=0.1284
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/shap/healthcare_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/Desk

...

  Best RMSE: 5.5248
  Params: {'n_estimators': 40, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 0.7, 'max_samples': 0.43341595973580715, 'n_jobs': -1, 'random_state': 42}
  RF - finance
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.4018     0.2581
  RMSE (%)                       4.772      5.525
  MAE (%)                        3.296      4.161
  Directional Acc (%)            69.44      63.77
  Base value (mean prediction) : 0.3689

  Saving evaluation results for /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results...
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results/finance_results.pkl
R² train=0.4018  test=0.2581
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/shap/finance_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/Desktop/mac

Best trial: 178. Best value: 4.97104: 100%|██████████| 200/200 [00:10<00:00, 19.27it/s]


  Best RMSE: 4.9710
  Params: {'n_estimators': 40, 'max_depth': 6, 'min_samples_split': 12, 'min_samples_leaf': 3, 'max_features': 0.5, 'max_samples': 0.7756629877372465, 'n_jobs': -1, 'random_state': 42}
  RF - industrial
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.5099     0.3246
  RMSE (%)                       3.685      4.971
  MAE (%)                        2.674      3.929
  Directional Acc (%)            79.37      68.12
  Base value (mean prediction) : 0.4949

  Saving evaluation results for /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results...
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results/industrial_results.pkl
R² train=0.5099  test=0.3246
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/shap/industrial_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/De

Best trial: 162. Best value: 9.6518: 100%|██████████| 200/200 [00:11<00:00, 17.03it/s] 


  Best RMSE: 9.6518
  Params: {'n_estimators': 70, 'max_depth': 4, 'min_samples_split': 11, 'min_samples_leaf': 3, 'max_features': 0.7, 'max_samples': 0.8802474677433052, 'n_jobs': -1, 'random_state': 42}
  RF - energy
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.4683     0.1350
  RMSE (%)                       4.511      9.652
  MAE (%)                        3.586      6.869
  Directional Acc (%)            71.03      50.72
  Base value (mean prediction) : 0.2785

  Saving evaluation results for /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results...
Saved evaluation  → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/results/energy_results.pkl
R² train=0.4683  test=0.1350
  Saved SHAP        → /Users/macbook/Desktop/macroeconomic-impact-stock-ml/models_sectors/rf/shap/energy_shap.pkl
    Features: 7
  Saved model       → /Users/macbook/Desktop/macroe

In [29]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

all_ridge_results = {}

for sector_name, return_col in SECTORS.items():

    print(f"\n{'='*55}")
    print(f"  RIDGE — {sector_name.upper()} → {return_col}")
    print(f"{'='*55}")


    # Split data
    splits = data_split(df, FEATURE_COLS, return_col)
    X_train = splits["X_train"]
    y_train = splits["y_train"]
    X_test  = splits["X_test"]
    y_test  = splits["y_test"]
    print(f"  Train: {len(X_train)} | Test: {len(X_test)}")

    # Find best alpha (same as previous model selection notebook)
    alphas = np.logspace(-3, 5, 100)

    cv_scores = []
    for alpha in alphas:
        ridge = Ridge(alpha=alpha)
        scores = cross_val_score(ridge, X_train, y_train, cv=5, scoring="r2")
        cv_scores.append(scores.mean())

    cv_scores = np.array(cv_scores)
    best_alpha = alphas[np.argmax(cv_scores)]
    print(f"  Best alpha: {best_alpha:.4f}")
    print(f"  Best CV R²: {cv_scores.max():.4f}")

    # Train final model
    model = Ridge(alpha=best_alpha)
    model.fit(X_train, y_train)

    # Evaluate
    ev_result = evaluate_model(
        model, X_train, y_train, X_test, y_test,
        f"Ridge - {sector_name}"
    )

    # Coefficients
    coef_df = pd.DataFrame({
        "Feature": FEATURE_COLS,
        "Coefficient": model.coef_,
        "Abs_Coef": np.abs(model.coef_),
    }).sort_values("Abs_Coef", ascending=False)
    print(f"\n  Coefficients:")
    print(coef_df.to_string(index=False))

    # SHAP
    shap_test_df, mean_abs_shap = compute_shap(
        model, X_train, X_test, FEATURE_COLS, model_type="linear"
    )

    # Save
    save_evaluation(sector_name, ev_result, FEATURE_COLS, y_test, RIDGE_MODEL_DIR)
    save_shap(sector_name, shap_test_df, mean_abs_shap, FEATURE_COLS, RIDGE_MODEL_DIR)
    save_model(sector_name, model, RIDGE_MODEL_DIR)

    all_ridge_results[sector_name] = {
        "best_alpha": best_alpha,
        "cv_r2": cv_scores.max(),
        "test_r2": ev_result[1]["r2"],
        "test_rmse": ev_result[1]["rmse"],
        "test_dir": ev_result[1]["dir"],
    }

print("\n ALL SECTORS TRAINED (RIDGE)")


  RIDGE — TECH → Tech_Return

  Features (7): CPI_Change_lag1, Rate_Change, GDP_Growth_lag2, Unemp_Change_lag1, USD_Change, VIX_Change, Credit_Spread_lag2
  Train: 252 | Test: 69
  Best alpha: 792.4829
  Best CV R²: 0.2557
  Ridge - tech
  Metric                         Train       Test
  ---------------------------------------------
  R²                            0.2424     0.1741
  RMSE (%)                       5.849      5.710
  MAE (%)                        4.177      4.406
  Directional Acc (%)            72.22      73.91

  Coefficients:
           Feature  Coefficient  Abs_Coef
        VIX_Change    -0.708301  0.708301
        USD_Change    -0.105842  0.105842
   CPI_Change_lag1     0.059099  0.059099
 Unemp_Change_lag1    -0.035071  0.035071
Credit_Spread_lag2    -0.034737  0.034737
   GDP_Growth_lag2     0.032444  0.032444
       Rate_Change     0.008030  0.008030
  Base value (mean prediction) : 0.3893

  Saving evaluation results for /Users/macbook/Desktop/macroeconomic-

In [30]:
summary = pd.DataFrame(all_rf_results).T
summary.columns = ["best_rmse", "test_r2", "test_rmse",'test_dir']
summary = summary.sort_values("test_r2", ascending=False)
print(summary.round(4).to_string())

            best_rmse  test_r2  test_rmse  test_dir
industrial     4.9710   0.3246     4.9710   68.1159
tech           5.3944   0.2629     5.3944   71.0145
finance        5.5248   0.2581     5.5248   63.7681
energy         9.6518   0.1350     9.6518   50.7246
healthcare     4.0093   0.1284     4.0093   63.7681


In [31]:
summary = pd.DataFrame(all_ridge_results).T
summary.columns = ["best_alpha", "cv_r2", "test_r2", "test_rmse", 'test_dir']

summary = summary.sort_values("test_r2", ascending=False)
print(summary.round(4).to_string())

            best_alpha   cv_r2  test_r2  test_rmse  test_dir
industrial    215.4435  0.2733   0.3581     4.8462   63.7681
finance       148.4968  0.2070   0.3330     5.2383   62.3188
energy        376.4936  0.2066   0.3316     8.4845   59.4203
tech          792.4829  0.2557   0.1741     5.7101   73.9130
healthcare    657.9332  0.2100  -0.0105     4.3170   65.2174


In [32]:
params_df = pd.DataFrame(all_rf_params).T
print("\nOptuna chose these hyperparameters per sector:")
print(params_df.to_string())

# save for reference
import pickle
import os

os.makedirs(RF_MODEL_PARAMS, exist_ok=True)
with open(f"{RF_MODEL_PARAMS}/all_params.pkl", "wb") as f:
    pickle.dump(all_rf_params, f)


Optuna chose these hyperparameters per sector:
            n_estimators  max_depth  min_samples_split  min_samples_leaf  max_features  max_samples  n_jobs  random_state
tech                90.0        3.0               10.0               8.0           0.7     0.614838    -1.0          42.0
healthcare          90.0        6.0                6.0               4.0           0.5     0.377555    -1.0          42.0
finance             40.0        5.0                5.0               3.0           0.7     0.433416    -1.0          42.0
industrial          40.0        6.0               12.0               3.0           0.5     0.775663    -1.0          42.0
energy              70.0        4.0               11.0               3.0           0.7     0.880247    -1.0          42.0
